# AG1: Actividad Guiada 1

**Nombre:** Evelyn Sánchez Laureano <br>

**Link Colab:** https://drive.google.com/file/d/17mVvzCAvU5xiP34GOAC4AKokTMJ-oqaW/view?usp=sharing <br>

Objetivos
- Comprender la utilidad y aplicar algoritmos divide y vencerás
- Comprender la utilidad y aplicar algoritmos voraces
- Comprender la utilidad y aplicar algoritmos de backtracking
- Realizar ejercicios avanzados → Búsqueda de distancias mínimas (opcional)

Divide y vencerás - Problema Torres de Hanói

- Parto de un problema y lo soluciono mediante la solución de subproblemas y unión de resultados.
- En este caso concreto: H(n) = H(n-1) + 1 + H(n-1)
- Pensar lo anterior como... “Muevo todos los discos salvo uno a torre auxiliar, muevo el disco más grande a la torre final y muevo todos los discos restantes de la torre auxiliar a la final”.
- Solamente resuelvo el problema de mover un disco (condición de parada de división) porque el resto se aplica el mismo algoritmo.
- Mover un disco es muy sencillo → Solución directa → Mover de torre de inicio a torre de fin.
- Ojo, el print del “if” es mover un disco entre torres (cualesquiera) y el print del “else” es el caso de mover el disco grande a torre final.

In [ ]:
import random, time, math
from functools import lru_cache

## 1. Torres de Hanoi – Divide y vencerás

Para mover $N$ fichas de `desde` a `hasta` se usa la torre auxiliar `6 - desde - hasta` (las torres son 1, 2, 3 y suman 6):
1. Mover $N-1$ fichas a la torre auxiliar (subproblema).
2. Mover la ficha mayor a su destino (caso simple).
3. Mover las $N-1$ fichas de la auxiliar al destino (subproblema).

Número de movimientos: $T(1)=1,\; T(n)=2T(n-1)+1 \Rightarrow T(n)=2^n-1$. **Complejidad $O(2^n)$.**

In [ ]:
def torres_hanoi(N, desde=1, hasta=3, movimientos=None):
    """Devuelve la lista de movimientos (desde, hasta) para mover N fichas."""
    if movimientos is None:
        movimientos = []
    if N == 1:
        movimientos.append((desde, hasta))
    else:
        pivote = 6 - desde - hasta
        torres_hanoi(N - 1, desde, pivote, movimientos)
        movimientos.append((desde, hasta))
        torres_hanoi(N - 1, pivote, hasta, movimientos)
    return movimientos

for d, h in torres_hanoi(3):
    print(f"Lleva la ficha desde {d} hasta {h}")

print()
for n in range(1, 11):
    m = len(torres_hanoi(n))
    print(f"N={n:2d}  movimientos={m:5d}  2^N-1={2**n-1:5d}  {'OK' if m == 2**n-1 else 'ERROR'}")

Lleva la ficha desde 1 hasta 3
Lleva la ficha desde 1 hasta 2
Lleva la ficha desde 3 hasta 2
Lleva la ficha desde 1 hasta 3
Lleva la ficha desde 2 hasta 1
Lleva la ficha desde 2 hasta 3
Lleva la ficha desde 1 hasta 3

N= 1  movimientos=    1  2^N-1=    1  OK
N= 2  movimientos=    3  2^N-1=    3  OK
N= 3  movimientos=    7  2^N-1=    7  OK
N= 4  movimientos=   15  2^N-1=   15  OK
N= 5  movimientos=   31  2^N-1=   31  OK
N= 6  movimientos=   63  2^N-1=   63  OK
N= 7  movimientos=  127  2^N-1=  127  OK
N= 8  movimientos=  255  2^N-1=  255  OK
N= 9  movimientos=  511  2^N-1=  511  OK
N=10  movimientos= 1023  2^N-1= 1023  OK


In [ ]:
# Comprobación: simulamos los movimientos y verificamos que nunca se pone una ficha mayor sobre una menor
def simula_hanoi(N):
    torres = {1: list(range(N, 0, -1)), 2: [], 3: []}
    for d, h in torres_hanoi(N):
        ficha = torres[d].pop()
        assert not torres[h] or torres[h][-1] > ficha, "Movimiento ilegal"
        torres[h].append(ficha)
    return torres

print(simula_hanoi(6))

{1: [], 2: [], 3: [6, 5, 4, 3, 2, 1]}


## 2. Sucesión de Fibonacci – recursión vs iteración

$F_n = F_{n-1} + F_{n-2}$, con $F_1 = F_2 = 1$.

* **Recursivo:** los subproblemas se solapan (para $F_6$ se calcula $F_4$ dos veces). Número de llamadas exponencial, $O(\varphi^n)\subseteq O(2^n)$. *Lo "raro" de la recursión*: se repiten cálculos y el tiempo explota a partir de $n \approx 30$.
* **Iterativo:** $O(n)$ en tiempo y $O(1)$ en espacio.
* **Memorización** (idea de programación dinámica): recursivo pero guardando resultados, $O(n)$.

> Nota: la versión de clase devolvía 1 para `N < 2`, lo que desplaza los índices (`Fibonacci(5)` = 8). Aquí se usa $F_1 = F_2 = 1$, así que $F_5 = 5$.

In [ ]:
LLAMADAS = 0
def fib_recursivo(n):
    global LLAMADAS
    LLAMADAS += 1
    if n <= 2:
        return 1
    return fib_recursivo(n - 1) + fib_recursivo(n - 2)

def fib_iterativo(n):
    a, b = 1, 1
    for _ in range(n - 2):
        a, b = b, a + b
    return 1 if n <= 2 else b

@lru_cache(maxsize=None)
def fib_memo(n):
    return 1 if n <= 2 else fib_memo(n - 1) + fib_memo(n - 2)

print([fib_iterativo(i) for i in range(1, 16)])

for n in (10, 20, 25, 30):
    LLAMADAS = 0
    t = time.perf_counter(); r = fib_recursivo(n); t_rec = time.perf_counter() - t
    t = time.perf_counter(); fib_iterativo(n); t_it = time.perf_counter() - t
    print(f"F({n})={r:7d}  llamadas recursivas={LLAMADAS:8d}  t_rec={t_rec:.4f}s  t_iter={t_it:.6f}s")

print("F(200) iterativo =", fib_iterativo(200))

[1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610]
F(10)=     55  llamadas recursivas=     109  t_rec=0.0000s  t_iter=0.000002s
F(20)=   6765  llamadas recursivas=   13529  t_rec=0.0018s  t_iter=0.000003s
F(25)=  75025  llamadas recursivas=  150049  t_rec=0.0208s  t_iter=0.000007s
F(30)= 832040  llamadas recursivas= 1664079  t_rec=0.2451s  t_iter=0.000009s
F(200) iterativo = 280571172992510140037611932413038677189525


## 3. Devolución de cambio – Técnica voraz

Elementos de la técnica voraz:
* **Candidatos:** los tipos de moneda del sistema.
* **Función de selección:** tomar la moneda de mayor valor posible sin pasarse (voracidad), por eso se recorre el sistema en orden decreciente.
* **Función de factibilidad:** el valor acumulado no supera la cantidad.
* **Función objetivo:** número total de monedas (a minimizar).
* **Criterio de solución:** el valor acumulado coincide con la cantidad.

Complejidad: $O(m)$ con $m$ = número de tipos de moneda (más $O(m\log m)$ si hay que ordenar).

In [ ]:
def cambio_monedas(cantidad, sistema):
    """Algoritmo voraz. Devuelve lista con el número de monedas de cada tipo
    (en el orden del sistema ordenado decrecientemente) o None si no hay solución."""
    sistema = sorted(sistema, reverse=True)
    solucion = [0] * len(sistema)
    acumulado = 0
    for i, valor in enumerate(sistema):
        monedas = (cantidad - acumulado) // valor   # función de selección
        solucion[i] = monedas
        acumulado += monedas * valor                # factibilidad
        if acumulado == cantidad:                   # criterio de solución final
            return solucion
    return None   # sin moneda unitaria puede no existir solución

print("Sistema [25,10,5,1], 27 ->", cambio_monedas(27, [25, 10, 5, 1]))
print("Sistema [25,10,5,1], 15 ->", cambio_monedas(15, [25, 10, 5, 1]))
print("Sistema [25,10,5],    3 ->", cambio_monedas(3, [25, 10, 5]))

Sistema [25,10,5,1], 27 -> [1, 0, 0, 2]
Sistema [25,10,5,1], 15 -> [0, 1, 1, 0]
Sistema [25,10,5],    3 -> None


### ¿Cuándo funciona y cuándo no?

La voracidad **no siempre da el óptimo**. Con el sistema $(11, 5, 1)$ y $C=15$ el voraz devuelve $11+1+1+1+1$ (5 monedas) cuando lo óptimo es $5+5+5$ (3 monedas).

Funciona en los sistemas llamados **canónicos** (euro, dólar: 1, 2, 5, 10, 20, 50…). Para comprobarlo comparamos con una solución exacta por programación dinámica, $\text{min}(c) = 1 + \min_{v \le c} \text{min}(c - v)$, de coste $O(C\cdot m)$. Un resultado conocido (Kozen y Zaks) dice que si el voraz falla, existe un contraejemplo menor que la suma de las dos monedas más grandes, así que basta con comprobar hasta ahí.

In [ ]:
def cambio_optimo(cantidad, sistema):
    """Programación dinámica: mínimo número de monedas para cada cantidad 0..C."""
    INF = float("inf")
    minimo = [0] + [INF] * cantidad
    for c in range(1, cantidad + 1):
        for v in sistema:
            if v <= c and minimo[c - v] + 1 < minimo[c]:
                minimo[c] = minimo[c - v] + 1
    return minimo[cantidad]

def contraejemplo_voraz(sistema):
    s = sorted(sistema, reverse=True)
    limite = s[0] + s[1] if len(s) > 1 else s[0]
    for c in range(1, limite + 1):
        v = cambio_monedas(c, s)
        n_voraz = sum(v) if v else float("inf")
        n_opt = cambio_optimo(c, s)
        if n_voraz != n_opt:
            return c, n_voraz, n_opt
    return None

for sistema in ([25, 10, 5, 1], [200, 100, 50, 20, 10, 5, 2, 1], [11, 5, 1], [4, 3, 1], [10, 6, 1]):
    r = contraejemplo_voraz(sistema)
    if r:
        print(f"{str(sistema):35s} NO canónico: C={r[0]} voraz={r[1]} monedas, óptimo={r[2]}")
    else:
        print(f"{str(sistema):35s} canónico: el voraz siempre es óptimo")

[25, 10, 5, 1]                      canónico: el voraz siempre es óptimo
[200, 100, 50, 20, 10, 5, 2, 1]     canónico: el voraz siempre es óptimo
[11, 5, 1]                          NO canónico: C=15 voraz=5 monedas, óptimo=3
[4, 3, 1]                           NO canónico: C=6 voraz=3 monedas, óptimo=2
[10, 6, 1]                          NO canónico: C=12 voraz=3 monedas, óptimo=2


## 4. N-Reinas – Vuelta atrás (backtracking)

**Modelo:** tupla $(x_1,\dots,x_N)$ donde $x_i$ es la fila de la reina de la columna $i$. Así, la restricción de columnas se cumple por construcción.

**Función de poda (`es_prometedora`):** la nueva reina de la columna `etapa` no comparte fila ($x_i \ne x_j$) ni diagonal ($|x_i - x_j| \ne |i-j|$) con las anteriores.

Versión de clase: comprueba toda la solución parcial en cada paso, $O(n^2)$ por nodo. Mejora: basta con comparar **solo la última reina** con las anteriores, $O(n)$ por nodo. La complejidad global sigue siendo exponencial, pero la poda reduce muchísimo los nodos visitados.

In [ ]:
def es_prometedora(solucion, etapa):
    """Solo hace falta comprobar la reina recién colocada contra las anteriores."""
    for i in range(etapa):
        if solucion[i] == solucion[etapa]:                         # misma fila
            return False
        if abs(solucion[i] - solucion[etapa]) == abs(i - etapa):   # misma diagonal
            return False
    return True

def reinas(N, solucion=None, etapa=0, soluciones=None):
    if solucion is None:
        solucion, soluciones = [0] * N, []
    for fila in range(1, N + 1):
        solucion[etapa] = fila
        if es_prometedora(solucion, etapa):
            if etapa == N - 1:
                soluciones.append(solucion.copy())
            else:
                reinas(N, solucion, etapa + 1, soluciones)
    solucion[etapa] = 0          # vuelta atrás
    return soluciones

def escribe(solucion):
    n = len(solucion)
    for fila in range(1, n + 1):
        print(" ".join("Q" if solucion[col] == fila else "·" for col in range(n)))

print("Soluciones 4 reinas:", reinas(4))
sol8 = reinas(8)
print(f"\n8 reinas: {len(sol8)} soluciones. Primera: {sol8[0]}")
escribe(sol8[0])

Soluciones 4 reinas: [[2, 4, 1, 3], [3, 1, 4, 2]]

8 reinas: 92 soluciones. Primera: [1, 5, 8, 6, 3, 7, 2, 4]
Q · · · · · · ·
· · · · · · Q ·
· · · · Q · · ·
· · · · · · · Q
· Q · · · · · ·
· · · Q · · · ·
· · · · · Q · ·
· · Q · · · · ·


In [ ]:
# Número de soluciones y tiempo frente al tamaño del tablero
for n in range(4, 11):
    t = time.perf_counter()
    s = reinas(n)
    print(f"N={n:2d}  soluciones={len(s):5d}  tiempo={time.perf_counter() - t:.3f}s")

N= 4  soluciones=    2  tiempo=0.000s
N= 5  soluciones=   10  tiempo=0.000s
N= 6  soluciones=    4  tiempo=0.000s
N= 7  soluciones=   40  tiempo=0.002s
N= 8  soluciones=   92  tiempo=0.010s
N= 9  soluciones=  352  tiempo=0.053s
N=10  soluciones=  724  tiempo=0.273s


## 5. Viaje por el río – Programación dinámica

$T(i,j)$ = tarifa directa de $i$ a $j$ (999 si no existe trayecto). $P(i,j)$ = precio óptimo de $i$ a $j$:

$$P(i,j) = \min\{\,T(i,j),\; P(i,k) + T(k,j)\ \ \forall\, i < k < j\,\}$$

Se cumple el principio de optimalidad de Bellman: si la ruta óptima de $i$ a $j$ pasa por $k$, el tramo $i\to k$ también es óptimo. Se guarda en `RUTA[i][j]` el último embarcadero intermedio para reconstruir el camino.

Complejidad: tres bucles anidados, **$O(n^3)$** en tiempo y $O(n^2)$ en espacio.

In [ ]:
INF = float("inf")
TARIFAS = [
    [0,   5,   4,   3,   INF, INF, INF],
    [INF, 0,   INF, 2,   3,   INF, 11 ],
    [INF, INF, 0,   1,   INF, 4,   10 ],
    [INF, INF, INF, 0,   5,   6,   9  ],
    [INF, INF, INF, INF, 0,   INF, 4  ],
    [INF, INF, INF, INF, INF, 0,   3  ],
    [INF, INF, INF, INF, INF, INF, 0  ],
]

def precios(tarifas):
    N = len(tarifas)
    P = [[INF] * N for _ in range(N)]
    RUTA = [[None] * N for _ in range(N)]
    for i in range(N):
        P[i][i] = 0
        RUTA[i][i] = i
        for j in range(i + 1, N):
            P[i][j] = tarifas[i][j]       # opción directa
            RUTA[i][j] = i
            for k in range(i + 1, j):     # opción con trasbordo en k
                if P[i][k] + tarifas[k][j] < P[i][j]:
                    P[i][j] = P[i][k] + tarifas[k][j]
                    RUTA[i][j] = k
    return P, RUTA

def calcular_ruta(RUTA, desde, hasta):
    """Reconstruye recursivamente la lista de embarcaderos de desde a hasta."""
    if desde == hasta:
        return [desde]
    return calcular_ruta(RUTA, desde, RUTA[desde][hasta]) + [hasta]

P, RUTA = precios(TARIFAS)
print("PRECIOS")
for fila in P:
    print(["-" if x == INF else x for x in fila])
print("\nRUTA")
for fila in RUTA:
    print(["" if x is None else x for x in fila])

ruta = calcular_ruta(RUTA, 0, 6)
print(f"\nRuta óptima 0 -> 6: {' -> '.join(map(str, ruta))}  (precio {P[0][6]})")

PRECIOS
[0, 5, 4, 3, 8, 8, 11]
['-', 0, '-', 2, 3, 8, 7]
['-', '-', 0, 1, 6, 4, 7]
['-', '-', '-', 0, 5, 6, 9]
['-', '-', '-', '-', 0, '-', 4]
['-', '-', '-', '-', '-', 0, 3]
['-', '-', '-', '-', '-', '-', 0]

RUTA
[0, 0, 0, 0, 1, 2, 5]
['', 1, 1, 1, 1, 3, 4]
['', '', 2, 2, 3, 2, 5]
['', '', '', 3, 3, 3, 3]
['', '', '', '', 4, 4, 4]
['', '', '', '', '', 5, 5]
['', '', '', '', '', '', 6]

Ruta óptima 0 -> 6: 0 -> 2 -> 5 -> 6  (precio 11)


In [ ]:
# Verificación por fuerza bruta: probar todos los caminos río abajo
def mejor_fuerza_bruta(t, i, j):
    if i == j:
        return 0
    return min(t[i][k] + mejor_fuerza_bruta(t, k, j) for k in range(i + 1, j + 1))

print("Fuerza bruta 0 -> 6:", mejor_fuerza_bruta(TARIFAS, 0, 6))
assert all(P[i][j] == mejor_fuerza_bruta(TARIFAS, i, j)
           for i in range(7) for j in range(i, 7))
print("La tabla de programación dinámica coincide con la fuerza bruta.")

Fuerza bruta 0 -> 6: 11
La tabla de programación dinámica coincide con la fuerza bruta.


## 6. Práctica individual – Par de puntos más cercanos

Dado un conjunto de puntos, encontrar los dos más cercanos.

| Versión | Idea | Complejidad |
|---|---|---|
| Fuerza bruta | comparar todos los pares | $O(n^2)$ |
| 1D ordenando | tras ordenar, el par más cercano es consecutivo | $O(n\log n)$ |
| 1D divide y vencerás | dividir por la mediana, combinar con el par frontera | $O(n\log n)$ |
| 2D/3D divide y vencerás | dividir por la mediana en $x$, revisar una franja de ancho $2\delta$ | $O(n\log^2 n)$ esta implementación, $O(n\log n)$ con mezcla |

In [ ]:
LISTA_1D = random.sample(range(1, 100000), 1000)   # valores distintos para que el resultado no sea trivialmente 0
LISTA_2D = [(random.randrange(1, 10000), random.randrange(1, 10000)) for _ in range(1000)]
LISTA_3D = [tuple(random.randrange(1, 10000) for _ in range(3)) for _ in range(1000)]

def distancia(a, b):
    if isinstance(a, (int, float)):
        return abs(a - b)
    return math.dist(a, b)

### 6.1 Fuerza bruta (sirve para 1D, 2D y 3D)
Se comparan los $\binom{n}{2} = \frac{n(n-1)}{2}$ pares: **$O(n^2)$**.

In [ ]:
def cercanos_fuerza_bruta(puntos):
    mejor = (INF, None, None)
    n = len(puntos)
    for i in range(n):
        for j in range(i + 1, n):
            d = distancia(puntos[i], puntos[j])
            if d < mejor[0]:
                mejor = (d, puntos[i], puntos[j])
    return mejor

print("1D:", cercanos_fuerza_bruta(LISTA_1D))
print("2D:", cercanos_fuerza_bruta(LISTA_2D))

1D: (1, 29255, 29256)
2D: (4.0, (9875, 7327), (9875, 7323))


### 6.2 1D con divide y vencerás
Se ordena una vez ($O(n\log n)$). Se divide la lista en dos mitades, se resuelve cada una y se **combina** comparando el mejor de cada lado con el par que cruza la frontera (último de la izquierda y primero de la derecha).

$T(n) = 2T(n/2) + O(1) \Rightarrow O(n)$ tras ordenar; en total **$O(n\log n)$**. (Equivale a recorrer los consecutivos de la lista ordenada.)

In [ ]:
def cercanos_1d_dyv(puntos):
    return _dyv_1d(sorted(puntos))

def _dyv_1d(p):
    n = len(p)
    if n <= 3:                                   # caso simple
        return cercanos_fuerza_bruta(p)
    mitad = n // 2
    izq = _dyv_1d(p[:mitad])                     # dividir
    der = _dyv_1d(p[mitad:])
    frontera = (p[mitad] - p[mitad - 1], p[mitad - 1], p[mitad])
    return min(izq, der, frontera, key=lambda x: x[0])   # combinar

print("1D D&V       :", cercanos_1d_dyv(LISTA_1D))
print("1D fuerza br.:", cercanos_fuerza_bruta(LISTA_1D))

1D D&V       : (1, 1231, 1232)
1D fuerza br.: (1, 29255, 29256)


### ¿Se puede mejorar el divide y vencerás en 1D?

En general, no. Encontrar el par más cercano tiene un límite inferior de $\Omega(n \log n)$ para cualquier algoritmo basado en comparaciones. Se demuestra reduciendo el problema de *unicidad de elementos* (¿hay dos números iguales en la lista?): si supiéramos hallar el par más cercano en menos de $n \log n$, podríamos resolver la unicidad mirando si la distancia mínima es 0, y para ese problema está demostrado que hacen falta $\Omega(n \log n)$ comparaciones.

Por tanto, tanto "ordenar y mirar los consecutivos" como el divide y vencerás son **asintóticamente óptimos** en este modelo:

| Versión | Complejidad |
|---|---|
| Fuerza bruta | $O(n^2)$ |
| Ordenar + consecutivos | $O(n \log n)$ |
| Divide y vencerás | $O(n \log n)$ (ordenar) $+\ O(n)$ (recursión) |
| Límite inferior (comparaciones) | $\Omega(n \log n)$ |

La única forma de bajar de ahí es **salir del modelo de comparaciones**. El algoritmo aleatorizado de Rabin (1976) usa una cuadrícula con *hashing* y consigue $O(n)$ en **tiempo esperado**, no en el peor caso.

### 6.3 2D y 3D con divide y vencerás
1. Ordenar por la coordenada $x$ y dividir por la mediana.
2. Resolver recursivamente cada mitad; $\delta$ = menor distancia encontrada.
3. **Combinar:** solo pueden mejorar $\delta$ los pares que cruzan la línea divisoria y están a menos de $\delta$ de ella (franja). Se ordena la franja por $y$ y cada punto se compara solo con los siguientes cuya diferencia en $y$ sea menor que $\delta$ (en 2D son como mucho 7 vecinos).

La misma función sirve para cualquier dimensión: en 3D la franja es una "loncha" de grosor $2\delta$ y se ordena por $y$; el número de vecinos a revisar sigue siendo acotado por una constante.

In [ ]:
def cercanos_dyv(puntos):
    return _dyv(sorted(puntos, key=lambda p: p[0]))

def _dyv(p):
    n = len(p)
    if n <= 3:
        return cercanos_fuerza_bruta(p)
    mitad = n // 2
    x_medio = p[mitad][0]
    mejor = min(_dyv(p[:mitad]), _dyv(p[mitad:]), key=lambda x: x[0])
    delta = mejor[0]
    franja = sorted((q for q in p if abs(q[0] - x_medio) < delta), key=lambda q: q[1])
    for i in range(len(franja)):
        j = i + 1
        while j < len(franja) and franja[j][1] - franja[i][1] < delta:
            d = distancia(franja[i], franja[j])
            if d < delta:
                delta, mejor = d, (d, franja[i], franja[j])
            j += 1
    return mejor

print("2D D&V       :", cercanos_dyv(LISTA_2D))
print("2D fuerza br.:", cercanos_fuerza_bruta(LISTA_2D))
print("3D D&V       :", cercanos_dyv(LISTA_3D))
print("3D fuerza br.:", cercanos_fuerza_bruta(LISTA_3D))

2D D&V       : (4.0, (9875, 7327), (9875, 7323))
2D fuerza br.: (4.0, (9875, 7327), (9875, 7323))
3D D&V       : (72.89718787443039, (6366, 5941, 3623), (6301, 5974, 3623))
3D fuerza br.: (72.89718787443039, (6366, 5941, 3623), (6301, 5974, 3623))


In [ ]:
# Validación con muchos casos aleatorios (se comparan distancias, puede haber empates)
for dim in (2, 3):
    for _ in range(200):
        pts = [tuple(random.randrange(0, 500) for _ in range(dim)) for _ in range(random.randint(2, 60))]
        assert abs(cercanos_dyv(pts)[0] - cercanos_fuerza_bruta(pts)[0]) < 1e-9
print("D&V coincide con la fuerza bruta en 400 casos aleatorios (2D y 3D).")

D&V coincide con la fuerza bruta en 400 casos aleatorios (2D y 3D).


### Complejidad en 2D y en 3D

2D. Recurrencia: $T(n) = 2T(n/2) + O(n \log n)$, porque en cada nivel se ordena la franja por $y$. Resolviéndola queda $O(n \log^2 n)$. Si en lugar de reordenar la franja se devuelven los puntos ya ordenados por $y$ desde la recursión (como en *merge sort*), la combinación pasa a $O(n)$ y el total queda en **$O(n \log n)$**.

El paso de combinación es eficiente gracias a un argumento geométrico: dentro de la franja, cada punto solo tiene que compararse con los puntos de un rectángulo $\delta \times 2\delta$, y en él caben **como mucho 8 puntos** separados al menos $\delta$ entre sí. Por eso cada punto se compara con un número constante de vecinos.

3D.La idea se generaliza: se divide por la mediana en $x$ y la franja pasa a ser una **loncha** de grosor $2\delta$. El argumento de empaquetado funciona igual en un prisma $\delta \times 2\delta \times 2\delta$, que solo admite un número constante de puntos separados al menos $\delta$. Con una implementación cuidadosa (resolver recursivamente el problema 2D dentro de la loncha) se llega a **$O(n \log n)$** en cualquier dimensión fija $d$ (Bentley y Shamos, 1976).

**Limitación de la implementación de este notebook.** La función cercanos_dyv revisa la loncha ordenando solo por $y$ y no filtra por $z$. El resultado es siempre correcto (validado en 400 casos aleatorios contra fuerza bruta), pero en el peor caso un punto podría compararse con muchos vecinos que están cerca en $x$ e $y$ y lejos en $z$. Con datos aleatorios esto no ocurre y el tiempo es prácticamente igual que en 2D, como muestra la tabla de tiempos.

### 6.4 Comparativa de tiempos
Al duplicar $n$, la fuerza bruta tarda ~4 veces más ($O(n^2)$) mientras que divide y vencerás tarda algo más del doble ($O(n\log n)$).

In [ ]:
print(f"{'n':>6} | {'fuerza bruta 2D':>16} | {'D&V 2D':>10} | {'D&V 3D':>10}")
for n in (500, 1000, 2000, 4000):
    p2 = [(random.random(), random.random()) for _ in range(n)]
    p3 = [(random.random(), random.random(), random.random()) for _ in range(n)]
    t = time.perf_counter(); cercanos_fuerza_bruta(p2); tfb = time.perf_counter() - t
    t = time.perf_counter(); cercanos_dyv(p2); tdv = time.perf_counter() - t
    t = time.perf_counter(); cercanos_dyv(p3); tdv3 = time.perf_counter() - t
    print(f"{n:6d} | {tfb:15.3f}s | {tdv:9.4f}s | {tdv3:9.4f}s")

     n |  fuerza bruta 2D |     D&V 2D |     D&V 3D
   500 |           0.052s |    0.0017s |    0.0025s
  1000 |           0.180s |    0.0034s |    0.0054s
  2000 |           0.935s |    0.0115s |    0.0178s
  4000 |           3.390s |    0.0201s |    0.0485s


## Conclusiones

* **Divide y vencerás** encaja cuando los subproblemas son disjuntos (Hanoi, par más cercano). Si se solapan (Fibonacci recursivo) la complejidad se dispara y conviene iterar o memorizar.
* **Voraz** es muy eficiente, pero solo garantiza el óptimo si el problema lo permite: en el cambio de monedas depende de que el sistema sea canónico.
* **Vuelta atrás** recorre el árbol de expansión; su eficiencia depende de una buena función de poda (en N-Reinas basta comprobar la última reina).
* **Programación dinámica** reutiliza soluciones parciales gracias al principio de optimalidad; en el viaje por el río pasa de un número exponencial de caminos a $O(n^3)$.